# GPT2 Small Experiments

## Introduction

My initial set of experiments involved implementing the MLA inspired extension for the attention output projection matrix $W^O$, to both the MHA and MLA attention variants and subsequently evaluating across a set of different $kv$, $q$ and $o$ compression dimensions for a reduced scale BERT and GPT2 models. While the results were promising across both the reduced scale BERT and GPT2 models, no trained model with compression outperformed their corresponding MHA baselines. Some variants did approach similar levels of performance, which is commendable at this scale especially given the majority of the models trainable weights are embedding parameters. As such I felt that it might be worth scaling the experiment to GPT2-small scale or larger. The main issue is this would most likely require the renting of external GPUs and could potentially be expensive. 

As such I wanted to explore the benefits of scaling via a cheaper proxy experiment before deciding whether to invest in scaling the model. The idea behind the first experiment is to take already trained GPT2 models of different sizes (small, medium and large) and replace their respective output projection matrices $W^O$ with an SVD reduced rank matrix. Essentially, I wish to replicate my experiment on these larger models without the need to pretrain from scratch. Therefore, for each layer in the respective model:

1. SVD is applied to that layers output projection matrix $W^O$, decomposing it into its singular values and singular vectors.
2. A set of $r$ fraction are defined, each is multiplied by matrix's full rank ($d_{model}$) to give the number of largest singular values to retain. These top $r$ singular values and their vectors form the best rank $r$ approxmation of $W^O$.
3. $W^O$ will then be replaced by this best rank $r$ approximation matrix, which can be stored as two smaller matrices of shape $d_{model} \times r$ and $r \times d_{model}$, directly mirroring the down and up projection matrices.

The idea being that this will approximate how my MLA inspired extention will function at a larger scale. This is not an exact replica of my MLA mechanism but should approximate the MHAE implementation, by testing if $W^O$ is low rank compressible on larger trained models.

After the matrix has been replaced by its best rank approximation matrix, the perplexity of the model will be evaluated on the `wikitext-103-raw-v1` validation dataset. Given none of these models have been trained with the best rank approximation matrices it is necessary to settle the new addition by continuing to pretrain the model on the `wikitext-103-raw-v1` training dataset. After a small number of steps, in this case 200, the perplexity will be evaluated again on the `wikitext-103-raw-v1` validation dataset.

A set of unchanged GPT2 small, medium and large models will also be evaluated on the `wikitext-103-raw-v1` validation dataset and will act as a set of baselines to compare against. 

Therefore, for each model the set of $r$ values [1.0, 0.5, 0.25, 0.125, 0.0625, 0.03125] will be analysed with perplexity captured both pre and post additional pretraining. These values will form a perplexity curve for each model. This will show how each model performs with higher and higher levels of compression, from 50% to 3.125% of the original rank (singular values) retained (where $r=1$ represents the baseline model). These curves will also be normalised by their respective baseline models perplexity value, therefore, allowing a true comparison across the different models. These normalised curves will allow me to compare the rate of increase in perplexity as the level of compression increases. 

At an individual model level, the positive result would be that the curves remain flat for as long as possible, suggesting that there is redundancy present in the $W^O$ matrix and that perplexity remains stable as the level of compression increases. However, I would expect to see some level of uptick in perplexity as the level of compression increases. Across curves, the ideal result would be that the rate of increase of the perplexity would differ across models, with larger models increasing more slowly than smaller models.

## Define the necessary libraries

In [1]:
# ! uv run pip install jupyter ipywidgets
# ! uv sync

In [2]:
# Base libraries
import random

# Maths
import math
import os
import torch
import pandas as pd
from tqdm.notebook import tqdm
from pathlib import Path
from omegaconf import DictConfig

# Transformers
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from transformers.utils import logging
from torch.optim import AdamW

# MLA
from mla.config.paths import get_pretraining_paths
from mla.model_training.model_training import ModelPreTraining
from mla.model_training.strategies import _PRETRAINING_STRATEGIES
from mla.utils.data_preparation import import_and_prepare_data, prepare_dataloaders
from mla.utils.setup import set_all_seeds

In [3]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [4]:
# Define env variables
os.environ["WANDB_SILENT"] = "true"
os.environ["TORCHDYNAMO_DISABLE"] = "1"

In [5]:
# Disable transformer logging 
logging.disable_progress_bar()

## General Configuration

In [6]:
# Define the config
config = {
    "seed": 42,
    "dataset_name": "Salesforce/wikitext",
    "dataset_config_name": "wikitext-103-raw-v1",
    "max_load_pct": None,
    "train_token_budget": None, 
    "val_token_budget": None,
    "sentence_keys": ["text"],
    "mlm": False,
    "mlm_probability": None,
    "root_dir": "data",
    
    "batch_size": 2,
    "drop_last": True,
    "eval_batch_size": 4,
    "gradient_accumulation_steps": 8,
    "mixed_precision": "fp16",
    "weight_decay": 0.01,
    "learning_rate": 5e-4,
    "keep_last_n_checkpoints": None,
    "experiment_name": None,
    "job_type": None,
    "eval_steps": 500,
    "eval_metric": "loss",
    "loss_type": "ForCausalLM",
    
    "task_type": "pretraining",
    "parallel_processes": 8,
    "num_workers": 0,
    "pin_memory": False,
    "hidden_size": None,
    "n_layer": None,
    "n_head": None,
    "max_seq_length": 1024, 
    "attention_mechanism": "MHA",
    "kv_compression_dim": None,
    "q_compression_dim": None,
    "output_compression_dim": None,
    "experiment_project": "GPT2_Testing",
    "r_thresholds": [1.0, 0.5, 0.25, 0.125, 0.0625, 0.03125],
    
    "num_labels": 0,
    "max_steps": 200,
    "warmup": True,
    "warmup_rate": 0.05,
    "max_norm": 1.0,
    "learning_rate": 3e-5,
    "weight_decay": 0.01,
    "train_eval_steps": 500,
    "eval_steps": 500,
}

## General Functions

### Apply SVD to each $W_O$ matrix

In [7]:
class OutputProjection(torch.nn.Module):
    """
    Define the replacement rank approximation output projection matrix. The matrices A and B are retained and not multiplied 
    together to ensure that the rank $r$ compression is respected. Otherwise, if A and B were multiplied back together we would 
    retain a matrix the same rank as the original $W^O$ matrix.
    """
    def __init__(self, A, B, bias):
        super().__init__()
        self.A = torch.nn.parameter.Parameter(A.detach())
        self.B = torch.nn.parameter.Parameter(B.detach())
        self.bias = torch.nn.parameter.Parameter(bias.detach())

    def forward(self, x):
        return ((x @ self.A) @ self.B) + self.bias

In [8]:
def keep_top_r(U, S, VT, r_threshold):
    """
    Retain the largest reduced_rank singular values and their associated singlar vectors. Combinating these singular values 
    and their corresponding left and right singlar vectors into two matrics which when combined approximate WO.
    """
    full_rank = len(S)
    reduced_rank = int(full_rank * r_threshold)
    s_sqrt = torch.sqrt(S[:reduced_rank])
    A = (U[:, :reduced_rank] * s_sqrt)
    B = (s_sqrt.unsqueeze(1) * VT[:reduced_rank])
    return A, B
    

In [9]:
def calculate_SVD_rank_metrics(WO, r_threshold):
    """
    First calculate the SVD and then generate the two components of the rank r approximation matrix. 
    """
    
    # Calculate the SVD
    U, S, VT = torch.linalg.svd(WO, full_matrices=False)

    # Calculate the retension based rank values
    A, B = keep_top_r(U, S, VT, r_threshold)

    return A, B

### Evaluate the model before continued pretraining

In [10]:
def eval_model_perplexity(model, validation_loader, root_dir, config):
    """
    Evaluate the models perplexity.
    """

    # Define the model pretraining paths
    paths = get_pretraining_paths(root_dir, config, config.seed)

    # Define the pretraining object
    pretraining = ModelPreTraining(
        model=model,
        optimizer=AdamW,
        metric_fn=_PRETRAINING_STRATEGIES[config.base_model].metric_cls(),
        config=config,
        paths=paths,
        seed=config.seed,
    )
    
    # Run the evaluation
    pretraining_results = pretraining.eval_model(validation_loader, training_eval=False)
    
    # Calculate the perplexity of the model
    model_loss = pretraining_results[0].avg
    model_perplexity = math.exp(model_loss)
    
    return model_loss, model_perplexity


### Continued Pretraining

In [11]:
def pretrain_model(model, train_loader, validation_loader, root_dir, config):
    """
    Continue to pretrain the model for 200 steps on the Wikitext training dataset.
    """

    # Define the model pretraining paths
    paths = get_pretraining_paths(root_dir, config, config.seed)

    # Define the pretraining object
    pretraining = ModelPreTraining(
        model=model,
        optimizer=AdamW,
        metric_fn=_PRETRAINING_STRATEGIES[config.base_model].metric_cls(),
        config=config,
        paths=paths,
        seed=config.seed,
    )

    # Continue to pretrain the model for 200 steps on the training dataset
    pretraining.model_training(training_dataloader=train_loader, validation_dataloader=validation_loader)

    # Evaluate the model again after continued pretraining
    pretraining_results = pretraining.eval_model(validation_loader, training_eval=False)

    # Calculate the perplexity of the model
    pt_loss = pretraining_results[0].avg
    pt_perplexity = math.exp(pt_loss)

    return pt_loss, pt_perplexity

### Evaluate the model before and after continued pretraining

In [12]:
def evaluate_gpt2_model(config):
    """
    Evaluate the GPT2 model before and after continued pretraining on 200 steps of WikiText V2 dataset.
    """
    # Define the results dictionary
    results_dict = {"model": [], "r": [], "actual_loss": [], "actual_ppl": [], "pretraining_loss": [], "pretraining_ppl": []}
    root_dir = Path(config.root_dir)
    
    # For each threshold
    for r_threshold in tqdm(config.r_thresholds):
    
        # Import the model
        model = GPT2LMHeadModel.from_pretrained(config.model_name, local_files_only=False)
    
        # Extract the WO weight matrices
        for i, layer in enumerate(model.transformer.h):
        
            # Identify the weight matrix
            bias = layer.attn.c_proj.bias
            WO = layer.attn.c_proj.weight
            
            # Calculate the entropy based effective rank and also retention based rank values
            A, B = calculate_SVD_rank_metrics(WO, r_threshold=r_threshold)
        
            # Update the model weights
            layer.attn.c_proj = OutputProjection(A, B, bias)
    
        # Evaluate the perplexity of the model
        actual_loss, actual_ppl = eval_model_perplexity(model, validation_loader, root_dir, config)
    
        # Continue to pretrain the model on a set of training data
        pretraining_loss, pretraining_ppl = pretrain_model(model, train_loader, validation_loader, root_dir, config)
    
        # Store the results
        results_dict["model"].append(config.model_name)
        results_dict["r"].append(r_threshold)
        results_dict["actual_loss"].append(actual_loss)
        results_dict["actual_ppl"].append(actual_ppl)
        results_dict["pretraining_loss"].append(pretraining_loss)
        results_dict["pretraining_ppl"].append(pretraining_ppl)

    return pd.DataFrame(results_dict)

## Load the WikiText datasets

In [13]:
def load_dataset(tokenizer, config):
    """
    Load the WikiText dataset and prepare dataloaders for the training and validation datasets.
    """
    # Ensure that it exists
    root_dir = Path(config.root_dir)
    root_dir.mkdir(parents=True, exist_ok=True)
    
    # Define the dataset paths
    data_paths = get_pretraining_paths(root_dir, config)
    
    # Prepare the datasets
    datasets = import_and_prepare_data(tokenizer, config, data_paths.tokenized_data_path)
    
    # Prepare the training dataloaders for continued pretraining
    train_loader, validation_loader = prepare_dataloaders(datasets, tokenizer, config, collator_fn=None)

    return train_loader, validation_loader

## GPT2 Small

In [14]:
# Wrap the config in a DictConfig
config = DictConfig(config)
config["base_model"] = "GPT2"
config["model_name"] = "gpt2"

In [15]:
# Set all seeds
set_all_seeds(config.seed)

In [16]:
# Define the TinyGPT2 model
tokenizer = GPT2Tokenizer.from_pretrained(config.base_model, local_files_only=False)

In [17]:
# Load the training and validation datasets
train_loader, validation_loader = load_dataset(tokenizer, config)

Loading Salesforce/wikitext/wikitext-103-raw-v1 dataset from: data/training_data/GPT2_Testing/pretraining/Salesforce__wikitext_wikitext-103-raw-v1_2b2a1bd31f



In [18]:
# Generate a set of results
gpt2_small_results = evaluate_gpt2_model(config)

  0%|          | 0/6 [00:00<?, ?it/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
Epoch 1:   2%|▌                            | 1197/57549 [02:21<1:45:14,  8.92it/s, loss=3.3235, step=149]

KeyboardInterrupt: 

## Save the results

In [ ]:
def save_results(df: pd.DataFrame, path: str = "data/svd_probe_results.csv") -> None:
    """Append a results DataFrame to a shared CSV (writes header only on first write)."""
    path = Path(path)
    write_header = not path.exists()
    df.to_csv(path, mode="a", header=write_header, index=False)

In [ ]:
# Save the results locally
save_results(gpt2_small_results)

## Conclusion

Initially I had wanted to run all experiments in one notebook, however, due to CUDA OOM's on my GPU I had to split the experiments out into seperate notebooks. The extending of the max_sequence_length has significantly increased the memory required. 